# 🚀 LSTM Customer Training - MEDIUM V2

**Configuración:** 120 días → 14 días (forecast intermedio)

**Plataforma:** Kaggle (GPU T4 x2) o Colab Pro

**Tiempo estimado:** ~3-4 horas

---

## ⚙️ Configuración Inicial

**Kaggle:**
1. Settings → Accelerator → **GPU T4 x2**
2. Add Data → Subir dataset

**Colab Pro:**
1. Runtime → Change runtime type → **GPU** (V100/A100)
2. Subir archivos manualmente

In [ ]:
# Verificar GPU
import tensorflow as tf
import gc

gc.collect()

print("="*80)
print("MEDIUM V2: 120→14 días")
print("="*80)
print(f"TensorFlow: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs: {gpus}")

if len(gpus) > 0:
    print(f"\n✅ {len(gpus)} GPU(s) DETECTADA(S)")
    for i, gpu in enumerate(gpus):
        print(f"   GPU {i}: {gpu.name}")
        tf.config.experimental.set_memory_growth(gpu, True)
    
    if len(gpus) > 1:
        strategy = tf.distribute.MirroredStrategy()
        print(f"\n🚀 Multi-GPU: {strategy.num_replicas_in_sync} GPUs")
else:
    print("\n⚠️ NO GPU - Activa GPU en Settings")

print("="*80)

In [ ]:
# Instalar dependencias
!pip install -q openpyxl seaborn
print("\n✅ Dependencias instaladas")
gc.collect()

In [ ]:
# Detectar plataforma y configurar rutas
import os
import sys

# Detectar si es Kaggle o Colab
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules

if IS_KAGGLE:
    print("📍 Plataforma: KAGGLE")
    BASE_PATH = '/kaggle/working'
    # Buscar dataset automáticamente
    datasets = [d for d in os.listdir('/kaggle/input/') if d != 'sample_submission']
    if datasets:
        DATASET_DIR = datasets[0]
        DATA_PATH = f'/kaggle/input/{DATASET_DIR}/online_retail_2.xlsx'
        SCRIPT_PATH = f'/kaggle/input/{DATASET_DIR}/train_all_customers_temporal_2.py'
        print(f"   Dataset: {DATASET_DIR}")
elif IS_COLAB:
    print("📍 Plataforma: COLAB")
    BASE_PATH = '/content'
    # Usuario subirá archivos
    from google.colab import files
    print("\n📤 Sube 'online_retail_2.xlsx'")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
    
    print("\n📤 Sube 'train_all_customers_temporal_2.py'")
    uploaded = files.upload()
    SCRIPT_PATH = list(uploaded.keys())[0]
else:
    print("📍 Plataforma: LOCAL")
    BASE_PATH = '.'
    DATA_PATH = 'data/processed/online_retail_2.xlsx'
    SCRIPT_PATH = 'src/train/train_all_customers_temporal_2.py'

print(f"\n✅ Configuración lista")
print(f"   Base: {BASE_PATH}")
print(f"   Data: {DATA_PATH}")
print(f"   Script: {SCRIPT_PATH}")

In [ ]:
# Copiar archivos
import shutil

os.makedirs(f'{BASE_PATH}/data/processed', exist_ok=True)
os.makedirs(f'{BASE_PATH}/models/temporal/customer_v2/medium', exist_ok=True)

if IS_KAGGLE:
    shutil.copy2(DATA_PATH, f'{BASE_PATH}/data/processed/online_retail_2.xlsx')
    shutil.copy2(SCRIPT_PATH, f'{BASE_PATH}/train_all_customers_temporal_2.py')
elif IS_COLAB:
    shutil.move(DATA_PATH, f'{BASE_PATH}/data/processed/online_retail_2.xlsx')
    shutil.move(SCRIPT_PATH, f'{BASE_PATH}/train_all_customers_temporal_2.py')

DATA_PATH = f'{BASE_PATH}/data/processed/online_retail_2.xlsx'

print("✅ Archivos copiados")
!ls -lh {BASE_PATH}/data/processed/
!ls -lh {BASE_PATH}/*.py
gc.collect()

In [ ]:
# Importar script
sys.path.append(BASE_PATH)

from train_all_customers_temporal_2 import CustomerTemporalAnalyzer, TemporalConfig

# Configuración optimizada
TemporalConfig.MEDIUM['batch_size'] = 32  # Balance velocidad/RAM

print("✅ Script importado")
print(f"📊 Config MEDIUM V2: {TemporalConfig.MEDIUM['window_days']}→{TemporalConfig.MEDIUM['forecast_days']}d")
print(f"   Batch: {TemporalConfig.MEDIUM['batch_size']}")
print(f"   Epochs: {TemporalConfig.MEDIUM['epochs']}")
gc.collect()

In [ ]:
# Preparar datos
import warnings
import numpy as np
from datetime import datetime
warnings.filterwarnings('ignore')

gc.collect()

start_time = datetime.now()
print(f"⏰ Inicio: {start_time}\n")

analyzer = CustomerTemporalAnalyzer(
    data_path=DATA_PATH,
    output_dir=f'{BASE_PATH}/models/temporal/customer_v2'
)

print("="*70)
print("FASE 1: Preparación de Datos")
print("="*70)

analyzer.load_and_preprocess_data()
gc.collect()

analyzer.calculate_rfm_metrics()
gc.collect()

analyzer.generate_customer_sequences(min_transactions=5)

# Optimización: Top 1500 clientes (suficiente para generalizar)
print(f"\n⚙️ Clientes: {len(analyzer.customers)} → 1500")
analyzer.customers = sorted(analyzer.customers, key=lambda x: x['TotalPurchases'], reverse=True)[:1500]

gc.collect()
print("\n✅ Datos listos")

In [ ]:
# ENTRENAR MEDIUM V2
import time

print("="*70)
print("FASE 2: Entrenamiento MEDIUM V2 (120→14 días)")
print("="*70)

t0 = time.time()

try:
    gc.collect()
    model, history, metrics = analyzer.train_horizon_model(TemporalConfig.MEDIUM)
    
    mins = (time.time() - t0) / 60
    
    print(f"\n✅ COMPLETADO ({mins:.1f} min)")
    print("="*70)
    print(f"Accuracy: {metrics['purchase_prob_accuracy']*100:.2f}%")
    print(f"AUC: {metrics['purchase_prob_auc']:.4f}")
    print(f"Days MAE: {metrics['days_mae']:.2f}")
    print(f"Value MAE: ${metrics['value_mae']:.2f}")
    print("="*70)
    
    # Guardar métricas en texto
    with open(f'{BASE_PATH}/models/temporal/customer_v2/medium/RESULTADOS.txt', 'w') as f:
        f.write(f"MEDIUM V2 - RESULTADOS\n")
        f.write(f"="*50 + "\n\n")
        f.write(f"Configuración: 120→14 días\n")
        f.write(f"Clientes: 1500\n")
        f.write(f"Batch: {TemporalConfig.MEDIUM['batch_size']}\n")
        f.write(f"Tiempo: {mins:.1f} min\n\n")
        f.write(f"Métricas:\n")
        f.write(f"  Accuracy: {metrics['purchase_prob_accuracy']*100:.2f}%\n")
        f.write(f"  AUC: {metrics['purchase_prob_auc']:.4f}\n")
        f.write(f"  Days MAE: {metrics['days_mae']:.2f}\n")
        f.write(f"  Value MAE: ${metrics['value_mae']:.2f}\n")
    
    del model, history
    gc.collect()
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()

total = (datetime.now() - start_time).total_seconds() / 60
print(f"\n⏰ Tiempo total: {total:.1f} min ({total/60:.1f}h)")

In [ ]:
# Comprimir modelos para descarga
!cd {BASE_PATH}/models/temporal && zip -r customer_v2_medium.zip customer_v2/medium/

print("✅ Modelos comprimidos")
print("📥 Descarga desde Output (Kaggle) o Files (Colab)")
!ls -lh {BASE_PATH}/models/temporal/*.zip

---

## 📊 Descarga de Resultados

**Kaggle:** Panel derecho → Output → Descargar `customer_v2_medium.zip`

**Colab:** Panel izquierdo → Files → Descargar archivo

**Descomprimir en:**
```
E:\Codigos\Proyecto Final\models\temporal\customer_v2\medium\
```